# Finance Agent Tutorial: Building an AI Agent with Claude Tool Use

In this interactive tutorial, you'll learn how to build a finance agent that uses Claude's tool use capabilities to:
- Look up stock prices
- Perform precise mathematical calculations
- Handle multi-turn conversations with an agentic loop

This notebook walks through each component step-by-step so you can understand how tool use works with Claude.

## Step 1: Setup and API Key Configuration

Import the required libraries and set up your Anthropic API key.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install anthropic python-dotenv

import json
import math
import os
from typing import Any, Dict, List, Optional
from getpass import getpass

from anthropic import Anthropic
from dotenv import load_dotenv

In [ ]:
# Load API key from .env file if it exists, otherwise prompt for it
load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    print("ANTHROPIC_API_KEY not found in environment.")
    print("Get your API key from: https://console.anthropic.com/")
    api_key = getpass("Enter your Anthropic API key: ")

# Validate API key format
if not api_key.startswith("sk-ant-"):
    raise ValueError(
        "Invalid API key format. API key should start with 'sk-ant-'. "
        "Please check your API key."
    )

# Initialize the Anthropic client
client = Anthropic(api_key=api_key)
print("✓ API key configured successfully!")

## Step 2: Configuration Constants

Define the configuration settings:
- **Model**: Claude 3.5 Haiku (fast and efficient)
- **Temperature**: 0.0 (deterministic for consistent JSON output)
- **Max tokens**: 1024 (sufficient for responses)

In [ ]:
# Claude API configuration
CLAUDE_MODEL = "claude-3-5-haiku-20241022"
MAX_TOKENS = 1024
TEMPERATURE = 0.0

print(f"Model: {CLAUDE_MODEL}")
print(f"Temperature: {TEMPERATURE} (deterministic)")
print(f"Max tokens: {MAX_TOKENS}")

## Step 3: Stock Price Data

This tutorial uses mock stock price data. In production, you would integrate with a real-time stock API.

**Try it yourself**: Add more stocks to the dictionary below!

In [ ]:
# Centralized stock price data - add new stocks here!
STOCK_PRICES: Dict[str, Dict[str, Any]] = {
    "aapl": {"price": 195.50, "name": "Apple"},
    "msft": {"price": 425.30, "name": "Microsoft"},
    "nvda": {"price": 875.20, "name": "NVIDIA"},
    "goog": {"price": 162.75, "name": "Alphabet (Google)"},
    "googl": {"price": 162.75, "name": "Alphabet (Google)"},
    "amzn": {"price": 185.40, "name": "Amazon"},
    "meta": {"price": 520.80, "name": "Meta (Facebook)"},
    "tsla": {"price": 245.60, "name": "Tesla"},
}

print(f"Available stocks: {len(STOCK_PRICES)}")
for ticker, data in STOCK_PRICES.items():
    print(f"  {ticker.upper()}: {data['name']} - ${data['price']}")

## Step 4: System Prompt

The system prompt guides Claude's behavior. It:
- Defines the agent's role
- Explains when to use each tool
- Encourages parallel tool calls for efficiency
- Enforces JSON output format

In [ ]:
SYSTEM_PROMPT = """You are a helpful finance assistant that can look up stock prices and perform precise calculations.

Always use the available tools for:
- Stock prices: Use calculate_portfolio_value to look up stock prices. For a single stock price, use quantity=1 (e.g., [{"ticker": "tsla", "quantity": 1}])
- Portfolio calculations: Use calculate_portfolio_value when asked about buying multiple stocks or calculating portfolio value
- Math calculations: Use calculate for any numerical computations to ensure precision

EFFICIENCY TIP - Parallel Tool Calls:
You can make multiple tool calls in a single response. When you identify operations that are independent (don't depend on each other's results), consider calling them in parallel to reduce the number of turns.

Example: For "173 * 3232 + 342 / 72.1", both the multiplication and division can be done in parallel since they're independent, then you can add the results.

When answering questions:
- For single stock price queries, use calculate_portfolio_value with [{"ticker": "XXX", "quantity": 1}]
- For portfolio questions (e.g., "buy 15 shares of tesla and 24 shares of google"), use calculate_portfolio_value with ALL stocks in a single call
- CRITICAL: Your final response MUST be ONLY valid JSON with no additional text before or after
- Use this exact format:
{"result": <number or string>, "explanation": "<brief optional explanation>"}

For numerical answers (stock prices, calculations), put just the number in "result".
For conversational questions (like asking about capabilities), put a text response in "result".
Do not include any commentary, just the JSON object."""

print("✓ System prompt configured")
print(f"\nPrompt length: {len(SYSTEM_PROMPT)} characters")

## Step 5: Tool Functions

Define the tool functions - the Python functions that Claude can call.

### Tool 1: Portfolio Value Calculator

This tool handles both single stock lookups and multi-stock portfolios.

In [ ]:
def calculate_portfolio_value(stocks: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Get stock prices and calculate portfolio values.
    
    Can be used for single stock lookups (quantity=1) or multiple stocks.

    Args:
        stocks: List of dicts, each with 'ticker' (str) and 'quantity' (int/float).
                For single stock price, use [{"ticker": "xxx", "quantity": 1}]

    Returns:
        Dict with breakdown per stock (including price per share) and total value

    Raises:
        KeyError: If any ticker is not found
    """
    breakdown = []
    total_value = 0.0

    for stock in stocks:
        ticker = stock["ticker"].lower()
        quantity = stock["quantity"]

        if ticker not in STOCK_PRICES:
            raise KeyError(f"Ticker '{ticker}' not found in available stocks")

        price = STOCK_PRICES[ticker]["price"]
        value = price * quantity

        breakdown.append(
            {
                "ticker": ticker,
                "name": STOCK_PRICES[ticker]["name"],
                "price": price,
                "quantity": quantity,
                "value": value,
            }
        )
        total_value += value

    return {"breakdown": breakdown, "total_value": total_value}

# Test the function
test_result = calculate_portfolio_value([{"ticker": "tsla", "quantity": 1}])
print("✓ calculate_portfolio_value function defined")
print(f"\nTest: Single TSLA stock")
print(json.dumps(test_result, indent=2))

### Tool 2: Mathematical Calculator

This tool performs precise binary operations.

In [ ]:
def calculate(op: str, a: float, b: float) -> Dict[str, Any]:
    """
    Perform a binary mathematical operation.

    Args:
        op: Operation to perform (+, -, *, /, **, log)
        a: First operand (left side, or base for **, or value for log)
        b: Second operand (right side, or exponent for **, or base for log)

    Returns:
        Dict with operation details and result for better context

    Raises:
        ValueError: If operation is not supported
        ZeroDivisionError: If division by zero
        ValueError: If logarithm with invalid base
    """
    match op:
        case "+":
            result = a + b
        case "-":
            result = a - b
        case "*":
            result = a * b
        case "/":
            if b == 0:
                raise ZeroDivisionError("Division by zero")
            result = a / b
        case "**":
            result = a**b
        case "log":
            if b <= 0 or b == 1 or a <= 0:
                raise ValueError(f"Invalid logarithm: log({a}, {b})")
            result = math.log(a, b)
        case _:
            raise ValueError(f"Unsupported operation: {op}")

    return {"operation": op, "a": a, "b": b, "result": result}

# Test the function
test_calc = calculate("*", 173, 3232)
print("✓ calculate function defined")
print(f"\nTest: 173 * 3232")
print(json.dumps(test_calc, indent=2))

## Step 6: Tool Specifications

Claude needs JSON schemas that describe the tools. These specifications tell Claude:
- What each tool does
- What parameters it accepts
- What type each parameter should be

This is how Claude decides when and how to use each tool.

In [ ]:
# Generate list of available tickers for the tool description
_portfolio_ticker_names = ", ".join(
    ["{} ({})".format(t, d["name"]) for t, d in STOCK_PRICES.items()]
)

calculate_portfolio_value_spec = {
    "name": "calculate_portfolio_value",
    "description": (
        "Get stock prices and calculate portfolio values. Use this for ANY stock price lookup - for a single stock, "
        "use quantity=1 (e.g., [{{\"ticker\": \"tsla\", \"quantity\": 1}}]). "
        "For multiple stocks, include all stocks in a single call. "
        f"Available tickers: {_portfolio_ticker_names}. Returns detailed breakdown with price per share and total value."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "stocks": {
                "type": "array",
                "description": "Array of stocks to calculate. Each item should have 'ticker' and 'quantity' fields.",
                "items": {
                    "type": "object",
                    "properties": {
                        "ticker": {
                            "type": "string",
                            "description": "Stock ticker symbol (lowercase, e.g., 'tsla', 'goog')",
                        },
                        "quantity": {
                            "type": "number",
                            "description": "Number of shares to buy",
                        },
                    },
                    "required": ["ticker", "quantity"],
                },
            }
        },
        "required": ["stocks"],
    },
}

calculate_spec = {
    "name": "calculate",
    "description": "Performs precise mathematical calculations. Supports binary operations on two numbers. Returns a JSON object with the operation, operands, and result for clear context.",
    "input_schema": {
        "type": "object",
        "properties": {
            "op": {
                "type": "string",
                "description": "The operation: '+' (add), '-' (subtract), '*' (multiply), '/' (divide), '**' (exponentiation/power), 'log' (logarithm)",
            },
            "a": {
                "type": "number",
                "description": "First number. For basic math (+, -, *, /): left operand. For '**': base. For 'log': the value. For sqrt, use '**' with a as the number and b as 0.5.",
            },
            "b": {
                "type": "number",
                "description": "Second number. For basic math (+, -, *, /): right operand. For '**': exponent. For 'log': the base (use 2.718281828459045 for natural log).",
            },
        },
        "required": ["op", "a", "b"],
    },
}

print("✓ Tool specifications defined")
print(f"\nTool 1: {calculate_portfolio_value_spec['name']}")
print(f"Tool 2: {calculate_spec['name']}")
print(f"\nExample tool description (portfolio):")
print(calculate_portfolio_value_spec['description'][:200] + "...")

## Step 7: Helper Functions

These utility functions work with Claude's responses.

In [ ]:
def parse_json_response(text: str) -> Dict[str, Any]:
    """
    Parse JSON from Claude's response.
    
    With proper prompting (system prompt specifies JSON format), Claude should
    return pure JSON. This function handles JSON parsing with a fallback if the response is not pure JSON.

    Args:
        text: The response text that should contain JSON

    Returns:
        Parsed JSON dictionary

    Raises:
        json.JSONDecodeError: If no valid JSON found
    """
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Fallback: try to find and parse JSON object if there's extra text
        start_idx = text.find("{")
        end_idx = text.rfind("}")
        if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
            try:
                return json.loads(text[start_idx : end_idx + 1])
            except json.JSONDecodeError:
                pass
        raise json.JSONDecodeError("No valid JSON found in response", text, 0)


def call_claude(messages: List[Dict[str, Any]], tools: Optional[List[Dict[str, Any]]] = None):
    """
    Call Claude API with tool use support.

    Args:
        messages: Conversation history
        tools: List of tool specifications (defaults to empty list)

    Returns:
        API response with content blocks
    """
    if tools is None:
        tools = []

    return client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        system=SYSTEM_PROMPT,
        tools=tools,
        messages=messages,
    )


def execute_tool_call(tool_call: Any) -> Dict[str, Any]:
    """
    Execute a single tool call based on its name and input.

    Args:
        tool_call: Tool call object with 'name' and 'input' attributes

    Returns:
        Dict result from the tool execution

    Raises:
        ValueError: If tool name is unknown or required parameters are missing
    """
    tool_name = getattr(tool_call, "name", None)
    tool_input = getattr(tool_call, "input", {})

    if tool_name == "calculate_portfolio_value":
        return calculate_portfolio_value(tool_input.get("stocks", []))
    elif tool_name == "calculate":
        op = tool_input.get("op")
        a = tool_input.get("a")
        b = tool_input.get("b")

        # Validate all required parameters are present
        if op is None or a is None or b is None:
            raise ValueError(
                f"Missing required parameters for calculate: op={op}, a={a}, b={b}"
            )

        return calculate(op, a, b)
    else:
        raise ValueError(f"Unknown tool: {tool_name}")

print("✓ Helper functions defined")

## Step 8: The Agentic Loop

This is the heart of the agent. The agentic loop:
1. Sends the user's prompt to Claude
2. Checks if Claude wants to use tools
3. Executes any requested tools
4. Returns results to Claude
5. Repeats until Claude provides a final answer

This multi-turn approach allows Claude to break down complex problems into steps.

In [ ]:
def finance_agent(prompt: str, verbose: bool = False) -> str:
    """
    Agent loop that handles multi-turn tool use to answer finance and math questions.

    The agent will:
    1. Send the user prompt to Claude with available tools
    2. Execute any tool calls requested by Claude
    3. Return tool results back to Claude
    4. Repeat until Claude provides a final text response

    Args:
        prompt: The user's question or request
        verbose: If True, print detailed information about each turn
        
    Returns:
        The final result as a string
    """
    messages = [{"role": "user", "content": prompt}]
    tools = [calculate_portfolio_value_spec, calculate_spec]
    
    turn = 0

    # Agent loop: continue until we get a response without tool calls
    while True:
        turn += 1
        if verbose:
            print(f"\n{'='*60}")
            print(f"Turn {turn}")
            print(f"{'='*60}")
        
        response = call_claude(messages, tools=tools)

        # Add Claude's response to the conversation
        messages.append({"role": "assistant", "content": response.content})

        # Check if Claude wants to use any tools
        tool_calls = [block for block in response.content if getattr(block, "type", None) == "tool_use"]

        if not tool_calls:
            # No more tool calls - extract and return the final answer
            final_answer = "\n".join([
                getattr(block, "text", "")
                for block in response.content
                if getattr(block, "type", None) == "text"
            ])

            if verbose:
                print(f"\n✓ Final response received (no tool calls)")
                print(f"Total turns: {turn}")

            # Parse JSON response (system prompt ensures JSON format)
            try:
                result_json = parse_json_response(final_answer)
                return str(result_json.get("result"))
            except json.JSONDecodeError:
                # Fallback: return raw response if JSON parsing fails
                return final_answer

        # Execute all tool calls and collect results
        if verbose:
            print(f"\n📞 Claude is calling {len(tool_calls)} tool(s):")
        
        tool_results = []
        for i, tool_call in enumerate(tool_calls, 1):
            tool_name = getattr(tool_call, "name", "unknown")
            if verbose:
                print(f"  {i}. {tool_name}")
                
            try:
                result = execute_tool_call(tool_call)
                content = json.dumps(result) if isinstance(result, dict) else str(result)
                if verbose:
                    print(f"     ✓ Success: {content[:100]}..." if len(content) > 100 else f"     ✓ Success: {content}")
            except (KeyError, ValueError, ZeroDivisionError) as e:
                content = json.dumps({"error": str(e)})
                if verbose:
                    print(f"     ✗ Error: {e}")

            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": tool_call.id,
                    "content": content,
                }
            )

        # Add all tool results to the conversation in a single user message
        messages.append({"role": "user", "content": tool_results})

print("✓ Agent function defined")

## Step 9: Test the Agent

Run the cells below to test the finance agent with various queries.

### Example 1: Simple Stock Price Lookup

In [ ]:
result = finance_agent("What's Tesla's stock price?", verbose=True)
print(f"\n📊 RESULT: {result}")

### Example 2: Mathematical Calculation

In [ ]:
result = finance_agent("Help me solve this math problem: 173 * 3232 + 342 / 72.1", verbose=True)
print(f"\n📊 RESULT: {result}")

### Example 3: Portfolio Calculation (Multiple Stocks)

In [ ]:
result = finance_agent(
    "How much would it cost to buy 15 shares of tesla, 24 shares of google, and 120 shares of amazon?",
    verbose=True
)
print(f"\n📊 RESULT: ${result}")

### Example 4: Complex Math with Functions

In [ ]:
result = finance_agent(
    "Help me solve this math problem: sqrt(234.13) + ln(27389140.25) + 173 * 32 + 4.5^2.",
    verbose=True
)
print(f"\n📊 RESULT: {result}")

### Example 5: Conversational Query

In [ ]:
result = finance_agent("Hi, what capabilities do you have?", verbose=True)
print(f"\n📊 RESULT: {result}")

## Step 10: Experiment with Your Own Queries

Try asking the agent your own questions.

In [ ]:
# Your custom query here!
my_query = "What's the price of Apple stock?"

result = finance_agent(my_query, verbose=True)
print(f"\n📊 RESULT: {result}")

## Key Takeaways

You've built a working AI agent with Claude tool use. Here's what you learned:

### 1. Tool Specifications
- Tools are defined as Python functions
- Tool specs use JSON schemas to describe inputs/outputs
- Clear descriptions help Claude decide when to use each tool

### 2. The Agentic Loop
- Claude can make multiple tool calls in a single response
- The agent loop continues until Claude provides a final answer
- This enables complex multi-step reasoning

### 3. Best Practices
- **Temperature 0.0**: Ensures consistent, deterministic output
- **System prompts**: Guide Claude's behavior and tool usage
- **Self-describing results**: Return structured data with context
- **Error handling**: Catch and format errors for Claude to handle
- **Parallel tool calls**: Independent operations can run together

### 4. Next Steps
You can extend this agent by:
- Adding more tools (news lookup, company info, etc.)
- Integrating real-time stock APIs
- Adding more complex calculations
- Implementing caching for better performance
- Adding streaming for real-time responses

## Additional Resources

- [Anthropic Tool Use Documentation](https://docs.claude.com/en/docs/agents-and-tools/tool-use)
- [Writing Tools for Agents](https://www.anthropic.com/engineering/writing-tools-for-agents)
- [Effective Context Engineering](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)